# Team B - EvolveGCN-H Training Notebook

This notebook converts the Team B implementation into `.ipynb` format and trains a temporal EvolveGCN-H model on preprocessed Elliptic snapshots.

In [1]:
import sys
print("Python executable:", sys.executable)

Python executable: /usr/local/bin/python3


In [2]:
print("Notebook execution test passed.")

Notebook execution test passed.


In [3]:
import json
import random
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import classification_report, f1_score

In [4]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def load_data(data_path: Path):
    return torch.load(data_path, weights_only=False)


def move_snapshots_to_device(snapshots: List, device: torch.device) -> List:
    return [data.to(device) for data in snapshots]


def compute_class_weights(snapshots: List, split_range: range, device: torch.device) -> torch.Tensor:
    y = torch.cat([snapshots[i].y for i in split_range], dim=0)
    licit = (y == 0).sum().item()
    illicit = (y == 1).sum().item()
    total = licit + illicit
    w0 = total / (2.0 * max(licit, 1))
    w1 = total / (2.0 * max(illicit, 1))
    return torch.tensor([w0, w1], dtype=torch.float32, device=device)


def detach_states(states: Optional[Tuple[torch.Tensor, torch.Tensor]]):
    if states is None:
        return None
    return tuple(s.detach() for s in states)

In [5]:
def precompute_gcn_norm(
    edge_index: torch.Tensor,
    n: int,
    device: torch.device,
    dtype: torch.dtype,
    add_self_loops: bool = True,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    if edge_index.numel() == 0:
        if add_self_loops:
            loop_idx = torch.arange(n, device=device)
            row = loop_idx
            col = loop_idx
        else:
            row = torch.empty(0, dtype=torch.long, device=device)
            col = torch.empty(0, dtype=torch.long, device=device)
    else:
        row = edge_index[0]
        col = edge_index[1]
        if add_self_loops:
            loop_idx = torch.arange(n, device=device)
            row = torch.cat([row, loop_idx], dim=0)
            col = torch.cat([col, loop_idx], dim=0)

    deg = torch.zeros(n, device=device, dtype=dtype)
    deg.index_add_(0, row, torch.ones(row.size(0), device=device, dtype=dtype))

    deg_inv_sqrt = deg.pow(-0.5)
    deg_inv_sqrt[torch.isinf(deg_inv_sqrt)] = 0.0
    norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]

    return row, col, norm


def cache_snapshot_graph_norm(snapshots: List) -> None:
    for data in snapshots:
        data._gcn_norm = precompute_gcn_norm(
            edge_index=data.edge_index,
            n=data.x.size(0),
            device=data.x.device,
            dtype=data.x.dtype,
            add_self_loops=True,
        )


def dense_gcn_propagation(
    x: torch.Tensor,
    edge_index: torch.Tensor,
    weight: torch.Tensor,
    precomputed_norm: Optional[Tuple[torch.Tensor, torch.Tensor, torch.Tensor]] = None,
    add_self_loops: bool = True,
) -> torch.Tensor:
    support = x @ weight

    if precomputed_norm is None:
        row, col, norm = precompute_gcn_norm(
            edge_index=edge_index,
            n=x.size(0),
            device=x.device,
            dtype=support.dtype,
            add_self_loops=add_self_loops,
        )
    else:
        row, col, norm = precomputed_norm

    if row.numel() == 0:
        return support if add_self_loops else torch.zeros_like(support)

    out = torch.zeros_like(support)
    out.index_add_(0, row, support[col] * norm.unsqueeze(1))
    return out


class WeightEvolver(nn.Module):
    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.hidden_size = in_channels * out_channels

        self.gru = nn.GRUCell(input_size=in_channels, hidden_size=self.hidden_size)
        self.h0 = nn.Parameter(torch.randn(self.hidden_size) * 0.05)

    def forward(
        self,
        node_embeddings: torch.Tensor,
        prev_hidden: Optional[torch.Tensor],
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        summary = node_embeddings.mean(dim=0)
        if prev_hidden is None:
            prev_hidden = self.h0
        next_hidden = self.gru(summary, prev_hidden)
        weight_t = next_hidden.view(self.in_channels, self.out_channels)
        return weight_t, next_hidden


class EvolveGCNHLayer(nn.Module):
    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        self.evolver = WeightEvolver(in_channels=in_channels, out_channels=out_channels)
        self.bias = nn.Parameter(torch.zeros(out_channels))

    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        prev_hidden: Optional[torch.Tensor],
        gcn_norm: Optional[Tuple[torch.Tensor, torch.Tensor, torch.Tensor]] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        weight_t, hidden_t = self.evolver(x, prev_hidden)
        out = dense_gcn_propagation(x, edge_index, weight_t, precomputed_norm=gcn_norm) + self.bias
        return out, hidden_t

In [6]:
class EvolveGCNH(nn.Module):
    def __init__(
        self,
        in_channels: int = 165,
        hidden1: int = 64,
        hidden2: int = 32,
        dropout: float = 0.5,
        num_classes: int = 2,
    ) -> None:
        super().__init__()
        self.layer1 = EvolveGCNHLayer(in_channels, hidden1)
        self.layer2 = EvolveGCNHLayer(hidden1, hidden2)
        self.classifier = nn.Linear(hidden2, num_classes)
        self.dropout = dropout

    def forward_snapshot(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        prev_states: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        gcn_norm: Optional[Tuple[torch.Tensor, torch.Tensor, torch.Tensor]] = None,
    ) -> Tuple[torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:
        if prev_states is None:
            h1_prev, h2_prev = None, None
        else:
            h1_prev, h2_prev = prev_states

        x1, h1_t = self.layer1(x, edge_index, h1_prev, gcn_norm=gcn_norm)
        x1 = F.relu(x1)
        x1 = F.dropout(x1, p=self.dropout, training=self.training)

        x2, h2_t = self.layer2(x1, edge_index, h2_prev, gcn_norm=gcn_norm)
        x2 = F.relu(x2)

        logits = self.classifier(x2)
        return logits, (h1_t, h2_t)


@dataclass
class Splits:
    train: range
    val: range
    test: range

In [7]:
def run_sequence(
    model: EvolveGCNH,
    snapshots: List,
    seq_range: range,
    device: torch.device,
    states: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
):
    logits_all = []
    labels_all = []

    for i in seq_range:
        data = snapshots[i]
        gcn_norm = getattr(data, "_gcn_norm", None)
        logits, states = model.forward_snapshot(data.x, data.edge_index, states, gcn_norm=gcn_norm)
        logits_all.append(logits)
        labels_all.append(data.y)

    return logits_all, labels_all, states


def evaluate_f1(
    model: EvolveGCNH,
    snapshots: List,
    eval_range: range,
    device: torch.device,
    warmup_range: Optional[range] = None,
):
    model.eval()
    states = None

    with torch.no_grad():
        if warmup_range is not None:
            _, _, states = run_sequence(model, snapshots, warmup_range, device, states)

        logits_all, labels_all, _ = run_sequence(model, snapshots, eval_range, device, states)

        y_true = torch.cat(labels_all, dim=0).cpu().numpy()
        y_pred = torch.cat([x.argmax(dim=1) for x in logits_all], dim=0).cpu().numpy()

    f1 = f1_score(y_true, y_pred, pos_label=1)
    return f1, y_true, y_pred


def train_epoch(
    model: EvolveGCNH,
    snapshots: List,
    split_range: range,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    grad_clip: float,
    tbptt_steps: int,
) -> float:
    model.train()
    states = None
    total_loss = 0.0

    for step, i in enumerate(split_range):
        data = snapshots[i]
        gcn_norm = getattr(data, "_gcn_norm", None)
        optimizer.zero_grad(set_to_none=True)

        logits, states = model.forward_snapshot(data.x, data.edge_index, states, gcn_norm=gcn_norm)
        loss = criterion(logits, data.y)
        loss.backward()

        if grad_clip > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        optimizer.step()
        total_loss += loss.item()

        # Prevent autograd from spanning across snapshot updates.
        if tbptt_steps <= 1:
            states = detach_states(states)
        elif (step + 1) % tbptt_steps == 0:
            states = detach_states(states)

    return total_loss / len(split_range)

In [8]:
import importlib.util
import subprocess
import sys


def ensure_torch_geometric() -> None:
    if importlib.util.find_spec("torch_geometric") is None:
        print("torch_geometric not found. Installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "torch-geometric"])
        print("torch_geometric installed. If the kernel still errors, restart the kernel and rerun.")
    else:
        print("torch_geometric is already installed.")


ensure_torch_geometric()

torch_geometric is already installed.


In [9]:
SEED = 42
DATA_PATH = Path("data/processed/snapshots.pt")
META_PATH = Path("data/processed/meta.json")
MODEL_PATH = Path("evolvegcn_h.pt")

EPOCHS = 10
LR = 1e-3
WEIGHT_DECAY = 1e-4
HIDDEN1 = 32
HIDDEN2 = 16
DROPOUT = 0.5
GRAD_CLIP = 1.0
TBPTT_STEPS = 1  # detach every step to keep memory stable
PATIENCE = 5
VAL_EVERY = 2
VAL_WARMUP_STEPS = 8

set_seed(SEED)
device = get_device()
print("Using device:", device)

snapshots = load_data(DATA_PATH)
print(f"Loaded {len(snapshots)} snapshots")

snapshots = move_snapshots_to_device(snapshots, device)
cache_snapshot_graph_norm(snapshots)
print("Snapshots moved to device and graph normalization cached.")

with open(META_PATH, "r", encoding="utf-8") as f:
    meta = json.load(f)

num_features = int(meta["num_features"])
splits = Splits(train=range(0, 34), val=range(34, 36), test=range(36, 49))

class_weights = compute_class_weights(snapshots, splits.train, device)
print("Class weights [licit, illicit]:", class_weights.tolist())

model = EvolveGCNH(
    in_channels=num_features,
    hidden1=HIDDEN1,
    hidden2=HIDDEN2,
    dropout=DROPOUT,
    num_classes=2,
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

Using device: mps
Loaded 49 snapshots
Snapshots moved to device and graph normalization cached.
Class weights [licit, illicit]: [0.5654888153076172, 4.317446708679199]


In [10]:
required_vars = [
    "EPOCHS", "model", "snapshots", "splits", "criterion", "optimizer",
    "device", "GRAD_CLIP", "TBPTT_STEPS", "PATIENCE", "MODEL_PATH",
    "VAL_EVERY", "VAL_WARMUP_STEPS"
]
missing = [name for name in required_vars if name not in globals()]

if missing:
    print(
        "Missing setup variables: " + ", ".join(missing) +
        ". Run Cell 14 first. On a fresh kernel, run Cells 7-14 before Cell 15."
    )
else:
    best_val = -1.0
    best_epoch = -1
    patience_ctr = 0

    warmup_start = max(splits.train.start, splits.train.stop - VAL_WARMUP_STEPS)
    val_warmup_range = range(warmup_start, splits.train.stop)

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_epoch(
            model=model,
            snapshots=snapshots,
            split_range=splits.train,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
            grad_clip=GRAD_CLIP,
            tbptt_steps=TBPTT_STEPS,
        )

        should_validate = (epoch == 1) or (epoch % VAL_EVERY == 0)
        if should_validate:
            val_f1, _, _ = evaluate_f1(
                model=model,
                snapshots=snapshots,
                eval_range=splits.val,
                device=device,
                warmup_range=val_warmup_range,
            )

            print(f"Epoch {epoch:03d} | Loss {train_loss:.4f} | Val F1(illicit) {val_f1:.4f}")

            if val_f1 > best_val:
                best_val = val_f1
                best_epoch = epoch
                patience_ctr = 0
                torch.save(model.state_dict(), MODEL_PATH)
                print(f"  -> New best model saved to {MODEL_PATH}")
            else:
                patience_ctr += 1

            if patience_ctr >= PATIENCE:
                print("Early stopping triggered.")
                break
        else:
            print(f"Epoch {epoch:03d} | Loss {train_loss:.4f} | Val skipped")

    print(f"Best validation F1: {best_val:.4f} at epoch {best_epoch}")

    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))

    test_f1, y_true, y_pred = evaluate_f1(
        model=model,
        snapshots=snapshots,
        eval_range=splits.test,
        device=device,
        warmup_range=range(0, 36),
    )
    print(f"Test F1(illicit): {test_f1:.4f}")
    print(classification_report(y_true, y_pred, digits=4, target_names=["licit", "illicit"]))

Epoch 001 | Loss 0.6840 | Val F1(illicit) 0.1981
  -> New best model saved to evolvegcn_h.pt
Epoch 002 | Loss 0.5677 | Val F1(illicit) 0.4226
  -> New best model saved to evolvegcn_h.pt
Epoch 003 | Loss 0.4749 | Val skipped
Epoch 004 | Loss 0.4522 | Val F1(illicit) 0.6353
  -> New best model saved to evolvegcn_h.pt
Epoch 005 | Loss 0.4170 | Val skipped
Epoch 006 | Loss 0.4248 | Val F1(illicit) 0.4734
Epoch 007 | Loss 0.3903 | Val skipped
Epoch 008 | Loss 0.3641 | Val F1(illicit) 0.6250
Epoch 009 | Loss 0.3599 | Val skipped
Epoch 010 | Loss 0.3590 | Val F1(illicit) 0.6296
Best validation F1: 0.6353 at epoch 4
Test F1(illicit): 0.3777
              precision    recall  f1-score   support

       licit     0.9664    0.9121    0.9384     12753
     illicit     0.2923    0.5334    0.3777       868

    accuracy                         0.8880     13621
   macro avg     0.6293    0.7228    0.6580     13621
weighted avg     0.9234    0.8880    0.9027     13621

